# Problem Set 4: Neural Networks (Supplementary Material)

## Overview

This file contains descriptions and implementation details of concepts like forward/backward propagation, convolution, and max-pooling, if we were to approach them by hand. You should rarely have to do them in this way, but it helps to understand what each step really means.

## Structure of This Notebook

This notebook contains the following chapters to further enhance your experience with PyTorch and deep learning fundamentals:
- **Chapter 1**: Working with PyTorch Tensors
- **Chapter 2**: Honing your PyTorch Skills
- **Chapter 3**: Concepts from Convolutional Neural Networks


### Imports

The following lines of code import packages and 
functions that are necessary for the following tasks.

If you like, you are encouraged to keep in mind which features call which imports, and to try to understand how they work. You can also look up the documentation for each of these imports to learn more about them.

As a reminder, please **do not** modify the following lines of code by adding,
removing or modifying the specified imports. 

In [ ]:
# RUN THIS CELL FIRST
import math
from collections import OrderedDict

import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision import datasets, transforms

import numpy as np
from numpy import allclose, isclose

from collections.abc import Callable

device = "cuda" if torch.cuda.is_available() else "cpu"
print(device)

# Extra Chapter 1 - Working with PyTorch tensors

### Concept 1.1 - What are Tensors?

Put simply, **tensors** are $n$-dimensional arrays that contain or represent information. 

In *PyTorch*, everything is defined as a [`torch.Tensor`](https://pytorch.org/docs/stable/tensors.html) type. 

 A `torch.Tensor` object in *PyTorch* looks like this:

![PyTorch](images/tensors.png)


---

Although [`torch.Tensor`](https://pytorch.org/docs/stable/tensors.html) objects are superficially very similar to NumPy arrays, (and indeed, they have many analogous interfaces), [`torch.Tensor`](https://pytorch.org/docs/stable/tensors.html) objects support PyTorch's powerful automatic differentiation engine as well as GPU acceleration, which makes them ideal for deep learning applications.

While it is easy to convert between NumPy arrays and [`torch.Tensor`](https://pytorch.org/docs/stable/tensors.html) objects in **MOST** cases, not all PyTorch functions accept NumPy arrays as input, and vice versa.

Try not to mix them up! 

![PyTorch](images/numpy_pytorch_diff.png)

If you are still curious about PyTorch, refer to this [website](https://pytorch-for-numpy-users.wkentaro.com/) for more information.


### Demo 1.1 - Tensor Basics

To create [`torch.Tensor`](https://pytorch.org/docs/stable/tensors.html) objects, you can use the [`torch.tensor(...)`](https://pytorch.org/docs/stable/generated/torch.tensor.html#torch.tensor) constructor:  

A 0-dimensional tensor: `torch.tensor(5.0)`  
A 1-dimensional tensor: `torch.tensor([1.0, 2.0, 3.0])`  
A 2-dimensional tensor: `torch.tensor([[.4, .3], [.1, .2]])`  

If automatic gradient computation is required, then the equivalent constructors will be:  
`torch.tensor(5.0, requires_grad=True)`  
`torch.tensor([1.0, 2.0, 3.0], requires_grad=True)`  
`torch.tensor([[.4, .3], [.1, .2]], requires_grad=True)`  

Notice that [`torch.Tensor`](https://pytorch.org/docs/stable/tensors.html) objects with `requires_grad=True` will be tracked for gradient computation, as evidenced by their `.grad` attribute (once you use them in some operations).

We can call `detach()` on them to stop them from being traced for gradient computation, returning us the tensors without `requires_grad=True`.

We can call `item()` on our tensors to return the value of any tensor with a single element as a standard Python number:

`>>> torch.tensor([1.0]).item()`

`1.0`

The following code block shows how we can make use of all these functions introduced.

In [ ]:
# Create a tensor with requires_grad set to True
x = torch.tensor([2.0], requires_grad=True)

# Compute the gradient of a simple expression using backward
y = x**2 + 2 * x
y.backward()

# Print the derivative value of y i.e dy/dx = 2x + 2 = 6.0.
print("Gradient of y with respect to x:", x.grad)

# Detach the gradient of x
x = x.detach()

# Print the gradient of x after detachment
print("Gradient of x after detachment:", x.grad)

# Extract the scalar value of a tensor as a Python number
x_value = x.item()
print("Value of x as a Python number:", x_value)

Note that the numbers which [`torch.Tensor`](https://pytorch.org/docs/stable/tensors.html) objects store have their own data type (e.g., `torch.float32`), and the data type of the output of an operation is determined by the data types of the input tensors. You can specify the data type of a tensor using the `dtype` argument in the constructor, or you can use the `.type(...)` method to change the data type of an existing tensor.

As can be imagined, integer tensors are **not** differentiable by definition, and thus, they cannot be tracked for gradient computation. 

In [ ]:
try:
    a = torch.tensor([1, 2, 3], dtype=torch.int8, requires_grad=True)
    b = a * a
except RuntimeError as e:
    print("Yup, integer tensors are not differentiable:")
    print(e)

### Demo 1.2 - Working with Tensors

PyTorch naturally offers a wide variety of arithmetic operations just as NumPy does. Here are some examples you can run:

In [ ]:
# Tensor-scalar operations
a1 = torch.tensor(50)
a2 = torch.tensor(75)
p = torch.tensor(2)

b = a1 + 4
print(b)  # torch.Tensor(54)

c = a1 - 4  
print(c)  # torch.Tensor(46)

d = a1 * 2 
print(d)  # torch.Tensor(100)

e = a2 / 2  
print(e)  # torch.Tensor(37.5)

f = a2 ** 2  
print(f)  # torch.Tensor(5625)

# Tensor-tensor operations

m1 = torch.tensor([[1, 2], [3, 4]], dtype=torch.float32)
m2 = torch.tensor([[5, 6], [7, 8]], dtype=torch.float32)

c = m1 + m2
print(c)  # torch.Tensor([[6, 8], [10, 12]])

d = m1 * m2
print(d)  # torch.Tensor([[5, 12], [21, 32]])

e = m1 / m2
print(e)  # torch.Tensor([[0.2, 0.3333], [0.4286, 0.5]])

f = m1 ** 2
print(f)  # torch.Tensor([[1, 4], [9, 16]])

g = m1 @ m2
print(g)  # torch.Tensor([[19, 22], [43, 50]])

h = torch.linalg.inv(m1)
print(h)  # torch.Tensor([[-2.0, 1.0], [1.5, -0.5]])


... and others you can read about:


Most, if not all, of these operations are differentiable by nature. This means you can use them within your *computation graph* and compute gradients.

| __Operation__                 | __Remarks__                                                                                                          |
|-------------------------------|----------------------------------------------------------------------------------------------------------------------|
| [`torch.sum(input)`](https://pytorch.org/docs/stable/generated/torch.sum.html)        | Returns the sum of all elements in the input tensor.                                                                 |
| [`torch.pow(base, exp)`](https://pytorch.org/docs/stable/generated/torch.pow.html)        | Returns the exponentiation of the base tensor to the exponent.                                                       |
| [`torch.mean(input)`](https://pytorch.org/docs/stable/generated/torch.mean.html)           | Returns the mean of all elements in the input tensor.                                                                |
| [`torch.square(input)`](https://pytorch.org/docs/stable/generated/torch.square.html)         | Returns the square of elements in the input tensor.                                                                  |
| [`torch.no_grad()`](https://pytorch.org/docs/stable/generated/torch.no_grad.html)             | Pauses all gradient computation and tracking inside the `with torch.no_grad()` block.                                |
| [`torch.matmul(input, other)`](https://pytorch.org/docs/stable/generated/torch.matmul.html)  | Returns the matrix product of tensors `input` and `other`. Same effect as `A @ B`.                                   |
| [`torch.reshape(input, shape)`](https://pytorch.org/docs/stable/generated/torch.reshape.html) | Returns the reshaped input matrix if dimensions commute. Same as `torch.view(input, shape)`.                         |
| [`torch.softmax(input, dim)`](https://pytorch.org/docs/stable/generated/torch.softmax.html)   | Computes the Softmax of an input along a specified dimension/axis.                                                   |
| [`torch.max(input, dim)`](https://pytorch.org/docs/stable/generated/torch.max.html)       | Returns the maximum element in the input tensor along a specific dimension/axis.                                     |
| [`torch.min(input, dim)`](https://pytorch.org/docs/stable/generated/torch.min.html)       | Returns the minimum element in the input tensor along a specific dimension/axis.                                     |
| [`torch.manual_seed(seed)`](https://pytorch.org/docs/stable/generated/torch.manual_seed.html)     | Sets the random number generator seed to the one specified. Good for reproducibility of runs.                        |
| [`torch.zeros(size)`](https://pytorch.org/docs/stable/generated/torch.zeros.html)           | Returns a tensor of zeros corresponding to the specific size.                                                        |
| [`torch.ones(size)`](https://pytorch.org/docs/stable/generated/torch.ones.html)            | Returns a tensor of ones corresponding to the specific size.                                                         |
| [`torch.squeeze(input, dim)`](https://pytorch.org/docs/stable/generated/torch.squeeze.html)   | Returns the tensor by removing a dimension `dim` from it. Eg: (1, 32, 32) -> dim=0 -> (32, 32)                       |
| [`torch.unsqueeze(input, dim)`](https://pytorch.org/docs/stable/generated/torch.unsqueeze.html) | Returns the tensor by adding an extra dimension at `dim`. Eg: (32, 32) -> dim=0 -> (1, 32, 32)                       |
| [`torch.clip(input, min, max)`](https://pytorch.org/docs/stable/generated/torch.clip.html) | Returns the tensor with all values in range `[min, max]`. All out-of-bounds values are made `max`/`min` accordingly. |

### Demo 1.2 - Randomness and Reproducibility

Most often, you are required to inject randomness to your experiments. Similar to `numpy`, you can generate [`torch.Tensor`](https://pytorch.org/docs/stable/tensors.html) objects of any arbitrary size/dimensionality with random values. Here are some ways to generate random tensors:

- `torch.rand(size)`: draws digits from Uniform distribution `x ~ U(0, 1)`
- `torch.randn(size)`: draws digits from Normal distribution `x ~ N(0, 1)`
- `torch.randint(low, high, size)`: generates tensors with random integers

As with NumPy, you can set the random seed to ensure reproducibility of your experiments. In PyTorch, you can do this with `torch.manual_seed(seed)`.

In [ ]:
torch.manual_seed(2109)
    
a = torch.rand(10, 10)  # a 10x10 matrix 
b = torch.rand(10)  # vector with 10 elements
c = torch.rand(10, 1)  # vector with 10 elements with an extra (insignificant) dimension
d = torch.rand(28, 28, 28)  # a "cube" tensor with 28 elements

e = torch.randn(10, 5)  # a 10x5 matrix

f = torch.randint(0, 100, (5, 5))  # a 5x5 matrix of integers in [0, 100)

### Demo 1.3 - Working with Tensors

Here, we use [`torch.linspace`](https://pytorch.org/docs/stable/generated/torch.linspace.html#torch.linspace) to create a [`torch.Tensor`](https://pytorch.org/docs/stable/tensors.html). In PyTorch (and Machine Learning in general) tensors form the basis of all operations.

We then make use of the built-in *PyTorch* function [`torch.sin`](https://pytorch.org/docs/stable/generated/torch.sin.html#torch.sin) to create the corresponding y-values of a sine function, and plot the points using *Matplotlib*.

In [ ]:
# This is a demonstration: You just need to run this cell without editing.

x = torch.linspace(-math.pi, math.pi, 1000) # Task 1.1: What is torch.linspace?
y_true = torch.sin(x)

plt.plot(x, y_true, linestyle='solid', label='sin(x)')
plt.axis('equal')
plt.title('Original function to fit')
plt.legend()
plt.show()

In [ ]:
# Run this cell to explore what the FIRST 10 VALUES of x has been assigned to.
# By default, each cell will always print the output of the last expression in the cell
# You can explore what x is by modifying the expression e.g. x.max(), x.shape
x[:10]

### Demo 1.4 - Using Tensors for linear regression

For this example, we fit a **degree 3 polynomial** to the sine function, using a learning rate of `1e-6` and `5000` iterations.

In [ ]:
# This is a demonstration: You just need to run this cell without editing.

# Set learning rate
learning_rate = 1e-6

# Initialize weights to 0
a = torch.tensor(0.)
b = torch.tensor(0.)
c = torch.tensor(0.)
d = torch.tensor(0.)

print('iter', 'loss', '\n----', '----', sep='\t')
for t in range(1, 5001): # 5000 iterations
    # Forward pass: compute predicted y
    y_pred = a + b * x + c * x**2 + d * x**3

    # Compute MSE loss
    loss = torch.mean(torch.square(y_pred - y_true))
    if t % 1000 == 0:
        print(t, loss.item(), sep='\t')

    # Backpropagation
    grad_y_pred = 2.0 * (y_pred - y_true) / y_pred.shape[0]
    
    # Compute gradients of a, b, c, d with respect to loss
    grad_a = grad_y_pred.sum()
    grad_b = (grad_y_pred * x).sum()
    grad_c = (grad_y_pred * x ** 2).sum()
    grad_d = (grad_y_pred * x ** 3).sum()

    # Update weights using gradient descent
    a -= learning_rate * grad_a
    b -= learning_rate * grad_b
    c -= learning_rate * grad_c
    d -= learning_rate * grad_d

# print fitted polynomial
equation = f'{a:.5f} + {b:.5f} x + {c:.5f} x^2 + {d:.5f} x^3'

y_pred = a + b * x + c * x**2 + d * x**3
plt.plot(x, y_true, linestyle='solid', label='sin(x)')
plt.plot(x, y_pred, linestyle='dashed', label=f'{equation}')
plt.axis('equal')
plt.title('3rd degree poly fitted to sine (MSE loss)')
plt.legend()
plt.show()

### Demo 1.5 - Using autograd to automatically compute gradients

In the previous example, we explicitly computed the gradient for Mean Squared Error (MSE):  
`grad_y_pred = 2.0 * (y_pred - y_true) / y_pred.shape[0]`

In the next example, we will use PyTorch's autograd functionality to help us compute the gradient for **Mean Absolute Error (MAE)**.  
In order to compute the gradients, we will use the `.backward()` method of *PyTorch* tensors.

Once again, we fit a **degree 3 polynomial** to the sine function, using a learning rate of `1e-6` and `5000` iterations.  
This time, we will use MAE instead of MSE.

In [ ]:
# This is a demonstration: You just need to run this cell without editing.

# Set learning rate
learning_rate = 1e-6

# Initialize weights to 0
a = torch.tensor(0., requires_grad=True)
b = torch.tensor(0., requires_grad=True)
c = torch.tensor(0., requires_grad=True)
d = torch.tensor(0., requires_grad=True)

print('iter', 'loss', '\n----', '----', sep='\t')
for t in range(1, 5001):
    # Forward pass: compute predicted y
    y_pred = a + b * x + c * x ** 2 + d * x ** 3

    # Compute MAE loss
    loss = torch.mean(torch.abs(y_pred - y_true))
    if t % 1000 == 0:
        print(t, loss.item(), sep='\t')

    # Automatically compute gradients
    loss.backward()

    # Update weights using gradient descent
    with torch.no_grad():
        a -= learning_rate * a.grad
        b -= learning_rate * b.grad
        c -= learning_rate * c.grad
        d -= learning_rate * d.grad
        a.grad.zero_() # reset gradients !important
        b.grad.zero_() # reset gradients !important
        c.grad.zero_() # reset gradients !important
        d.grad.zero_() # reset gradients !important
        # What happens if you don't reset the gradients?

# print fitted polynomial
equation = f'{a:.5f} + {b:.5f} x + {c:.5f} x^2 + {d:.5f} x^3'

y_pred = a + b * x + c * x ** 2 + d * x ** 3
plt.plot(x, y_true, linestyle='solid', label='sin(x)')
plt.plot(x, y_pred.detach().numpy(), linestyle='dashed', label=f'{equation}')
plt.axis('equal')
plt.title('3rd degree poly fitted to sine (MAE loss)')
plt.legend()
plt.show()

### Demo 1.6 - Polyfit model

We have demonstrated how to fit a degree-3 polynomial to a set of `x` and `y` points (following the sine curve), using two different types of loss functions (MSE and MAE).  

Now, we create a function `polyfit` that takes in some arbitrary set of points, and iteratively conducts backpropagation and weight update.
1. `x`, corresponding x-values,  
2. `y`, corresponding true y-values,  
3. `loss_fn` to compute the loss, given the true `y` and predicted `y`,  
4. `n` representing the $n$-degree polynomial, and 
5. `lr` learning rate, and  
6. `n_iter` for the number of times to iterate.  

For example,
```
>>> y = torch.sin(x)
>>> mse = lambda y_true, y_pred: torch.mean(torch.square(y_pred - y_true))
>>> mae = lambda y_true, y_pred: torch.mean(torch.abs(y_pred - y_true))

>>> polyfit(x, y, mse, 3, 1e-3, 5000)
tensor([-4.2270e-09,  8.5167e-01,  1.2131e-08, -9.2587e-02], requires_grad=True))

>>> polyfit(x, y, mae, 3, 1e-3, 5000)
tensor([-9.6776e-07,  8.7905e-01, -2.4784e-06, -9.8377e-02], requires_grad=True))
```

In [ ]:

def polyfit(x: torch.Tensor, y: torch.Tensor, loss_fn: Callable, n: int, lr: float, n_iter: int):
    """
    Parameters
    ----------
        x : A tensor of shape (1, n)
        y : A tensor of shape (1, n)
        loss_fn : Function to measure loss
        n : The nth-degree polynomial
        lr : Learning rate
        n_iter : The number of iterations of gradient descent
        
    Returns
    -------
        Near-optimal coefficients of the nth-degree polynomial as a tensor of shape (1, n+1) after `n_iter` epochs.
    """
    # Generate polynomial values
    poly_x = x ** torch.arange(0, n+1)[:, None]

    # Initialize weights to 0
    coefficients = torch.zeros(n+1, requires_grad=True)

    for _ in range(n_iter):
        y_pred = coefficients @ poly_x
        loss = loss_fn(y, y_pred)
        loss.backward()
        with torch.no_grad():
            coefficients -= lr * coefficients.grad
            coefficients.grad.zero_()
    return coefficients


x = torch.linspace(-math.pi, math.pi, 1000)

# Original true values
y = torch.sin(x)
plt.plot(x, y, linestyle='solid', label='sin(x)')

# MSE
mse = lambda y_true, y_pred: torch.mean(torch.square(y_pred - y_true))
a, b, c, d = polyfit(x, y, mse, 3, 1e-6, 5000)
y_pred_mse = a + b * x + c * x ** 2 + d * x ** 3
plt.plot(x, y_pred_mse.detach().numpy(), linestyle='dashed', label=f'mse')

# MAE
mae = lambda y_true, y_pred: torch.mean(torch.abs(y_pred - y_true))
a, b, c, d = polyfit(x, y, mae, 3, 1e-3, 5000)
y_pred_mae = a + b * x + c * x ** 2 + d * x ** 3
plt.plot(x, y_pred_mae.detach().numpy(), linestyle='dashed', label=f'mae')

plt.axis('equal')
plt.title('Comparison of different fits')
plt.legend()
plt.show()

---
### Computing gradients for arbitrary graphs

Recall the neural network for `y = |x-1|` from the lecture. We are going to implement forward propagation as mentioned during lecture. This forward pass is the act of feeding data into our input layer, which will then be passed to and processed by the hidden layers according to the different activation functions specific to each perceptron. After passing through all the hidden layers, our neural network will generate an output, $\hat{y}$, that is hopefully meaningful to our problem at hand.

<div>
<img src="images/toy_nn_old.png" width=500>
</div>

### Demo 1.7 - Implementing Forward Pass
The function `forward_pass` that takes in 4 arguments:  
1. `x`, the input values (not including bias)
2. `w0`, (2x2) weights of the hidden layer
3. `w1`, (3x1) weights of the output layer
4. `activation_fn`, the activation function of the hidden layer.

*Note: As in the lecture, there will be no activation for the output layer (i.e. the activation function of the output layer is the identity function `lambda x: x`)*

In [ ]:
def forward_pass(x: torch.Tensor, w0: torch.Tensor, w1: torch.Tensor, activation_fn: Callable):
    x_with_bias = torch.hstack((torch.ones(x.shape[0], 1), x))
    a = activation_fn(x_with_bias @ w0)
    a_with_bias = torch.hstack((torch.ones(a.shape[0], 1), a))
    y_pred = a_with_bias @ w1
    return y_pred

# Exact weights
w0 = torch.tensor([[-1., 1.], [1., -1.]], requires_grad=True)
w1 = torch.tensor([[0.], [1.], [1.]], requires_grad=True)

# Performing a forward pass on exact solution for weights will give us the correct y values
x_sample = torch.linspace(-2, 2, 5).reshape(-1, 1)
forward_pass(x_sample, w0, w1, torch.relu) # tensor([[3.], [2.], [1.], [0.], [1.]])

### Demo 1.8 - Backward propagation
Now, we can start with random weights for `w0` and `w1`, and iteratively perform forward passes and backward propagation multiple times to converge on a solution.

In [ ]:
x = torch.linspace(-10, 10, 1000).reshape(-1, 1)
y = torch.abs(x-1)
torch.manual_seed(1) # Set seed to some fixed value
w0 = torch.randn(2, 2, requires_grad=True)
w1 = torch.randn(3, 1, requires_grad=True)
learning_rate = 0.023
print('iter', 'loss', '\n----', '----', sep='\t')
for t in range(1, 10001):
    # Forward pass: compute predicted y
    y_pred = forward_pass(x, w0, w1, torch.relu)

    loss = torch.mean(torch.square(y - y_pred))
    loss.backward()

    if t % 1000 == 0:
        print(t, loss.item(), sep='\t')

    with torch.no_grad():
        """
        Update weights and then reset the gradients to zero here
        """
        
y_pred = forward_pass(x, w0, w1, torch.relu)
loss = torch.mean(torch.square(y - y_pred)).item()
print("--- w0 ---", w0, sep='\n')
print("--- w1 ---", w1, sep='\n')
plt.plot(x, y, linestyle='solid', label='|x-1|')
plt.plot(x, y_pred.detach().numpy(), linestyle='dashed', label='perceptron')
plt.axis('equal')
plt.title('Fit NN on abs function')
plt.legend()
plt.show()

# Extra Chapter 2 - Honing Your PyTorch Skills

PyTorch is a powerful library where deep learning is concerned. Unfortunately, that also means it is a very big library, and it can be overwhelming to know where to start.

We give you a few more exhibits of PyTorch's capabilities here, and we encourage you to explore the documentation and other resources to learn more about it.

### Listicle 2.1 - Common torch.nn Layers


*Note: If you haven't done Task 1.1 in the problem set proper already, you may want to check it out before going through the following listicles. It is likely that the listicles here would not make sense without the context of Task 1.1.*

PyTorch's [`torch.nn`](https://pytorch.org/docs/stable/nn.html) module contains a wide variety of layers that are commonly used in deep learning. Here are some examples you can explore:


| Layer                   | Usage                                                                | Remarks                                                                                                  |
|-------------------------|----------------------------------------------------------------------|----------------------------------------------------------------------------------------------------------|
| Fully-connected / Dense | [`nn.Linear(in_features, out_features, bias=True)`](https://pytorch.org/docs/stable/generated/torch.nn.Linear.html)                  | Inputs are vectors of size `in_features`. Performs `Y=Wx+b` and outputs a vector of size `out_features`. |
| Convolution | [`nn.Conv2d(in_channels, out_channels, kernel_size, stride, padding)`](https://pytorch.org/docs/stable/generated/torch.nn.Conv2d.html) | Inputs are images/tensors with `in_channels` number of channels of arbitrary height and width.           |
| ReLU                    | [`nn.ReLU()`](https://pytorch.org/docs/stable/generated/torch.nn.ReLU.html)                                                          | Performs the Rectified Linear Units (ReLU) activation on the input tensor.                               |
| Leaky ReLU              | [`nn.LeakyReLU(negative_slope=0.01)`](https://pytorch.org/docs/stable/generated/torch.nn.LeakyReLU.html)                                  | Performs the Leaky ReLU activation on the input tensor with the specified negative slope.                |
| Sigmoid                 | [`nn.Sigmoid()`](https://pytorch.org/docs/stable/generated/torch.nn.Sigmoid.html)                                                       | Performs the Sigmoid activation on the input tensor with the specified negative slope.                   |
| Max Pooling             | [`nn.MaxPool2d(pool_size)`](https://pytorch.org/docs/stable/generated/torch.nn.MaxPool2d.html)                                            | Performs the Maximum Pooling operation on the input tensor with the specified pooling size.              |
| Dropout                 | [`nn.Dropout(p)`](https://pytorch.org/docs/stable/generated/torch.nn.Dropout.html)                                                      | Performs Dropout on the layer _prior to being called_ with the specified dropping probability.           |

### Listicle 2.2 - Loss Functions

PyTorch's [`torch.nn`](https://pytorch.org/docs/stable/nn.html) module also contains a wide variety of loss functions that are commonly used in deep learning. Here are some examples you can explore:

| **Loss**               | **Usage**               |
|------------------------|-------------------------|
| Cross Entropy          | [`nn.CrossEntropyLoss()`](https://pytorch.org/docs/stable/generated/torch.nn.CrossEntropyLoss.html) |
| Binary Cross Entropy   | [`nn.BCELoss()`](https://pytorch.org/docs/stable/generated/torch.nn.BCELoss.html)          |
| Mean Squared Error     | [`nn.MSELoss()`](https://pytorch.org/docs/stable/generated/torch.nn.MSELoss.html)          |
| Mean Absolute Error    | [`nn.L1Loss()`](https://pytorch.org/docs/stable/generated/torch.nn.L1Loss.html)           |
| Negative Log Likelihood| [`nn.NLLLoss()`](https://pytorch.org/docs/stable/generated/torch.nn.NLLLoss.html)         |

After computing the output of the forward pass using your model, you can do:

```python
loss_fn = nn.XYZLoss()  # some arbitary loss from the above table

output = ...  # some tensor
target = ...  # some tensor
loss = loss_fn(output, target)

"""
As mentioned above, to backpropagate the loss wrt the parameters, you can simply call
`loss.backward()` and it will compute the partial derivates (i.e., the gradients) and
store them inside the `.grad` attribute of each and every parameter tensor!
"""
```

### Listicle 2.3 - Optimizers

*Note: As above, you may want to have done Task 1.2 before going through the following listicle.*

PyTorch's [`torch.optim`](https://pytorch.org/docs/stable/optim.html) module contains a wide variety of optimizers that are commonly used in deep learning. 

These optimizers generalize the process of `Stochastic Gradient Descent` (SGD) by implementing various alternative optimization algorithms that can be used to update the parameters of a model based on the computed gradients.

| **Optimiser**                     | **Usage**                                  |
|-----------------------------------|--------------------------------------------|
| [Stochastic Gradient Descent (SGD)](https://pytorch.org/docs/stable/generated/torch.optim.SGD.html) | [`torch.optim.SGD(parameters, lr)`](https://pytorch.org/docs/stable/generated/torch.optim.SGD.html)          |
| [Adaptive Momentum (Adam)](https://pytorch.org/docs/stable/generated/torch.optim.Adam.html)          | [`torch.optim.Adam(parameters, lr=0.001)`](https://pytorch.org/docs/stable/generated/torch.optim.Adam.html)   |
| [Adaptive Gradient (Adagrad)](https://pytorch.org/docs/stable/generated/torch.optim.Adagrad.html)       | [`torch.optim.Adagrad(parameters, lr=0.01)`](https://pytorch.org/docs/stable/generated/torch.optim.Adagrad.html) |

# Extra Chapter 3 - Concepts from Convolutional Neural Networks
 
### What Convolution Does

**Convolution** is a mathematical operation that applies a small matrix, called a **filter** (or **kernel**), across every part of an input image or feature map.

At each step, the filter multiplies its values element-wise with the corresponding section of the input and sums the results, producing a single pixel value in the **output feature map**. This process is repeated by sliding the filter (the "convolution") across the entire input, effectively creating a compressed map of detected features.

### Why We Do It

We use convolution as the quintessential core of a CNN for two critical reasons:

1.  **Feature Extraction:** The filter acts as a feature detector. Different filters learn to recognize specific local patterns, such as **edges, curves, or textures**. Applying multiple unique filters in a layer allows the network to automatically and simultaneously identify many different low-level features across the entire image.

2.  **Efficiency and Robustness (Parameter Sharing):** Since the **exact same filter is reused** (weights are "shared") across the entire input, the model learns that a feature is important regardless of where it appears in the image. This dramatically **reduces the number of parameters** compared to a fully connected layer, making the network much faster to train and less prone to overfitting.

***

Here, we implement the `conv2d` function that performs the convolution operation on a 2-dim image, `img : torch.Tensor`, using a certain kernel, `kernel : torch.Tensor`. We assume there is no padding and the stride is 1. 

We can run an example to see what it does to two given two images `x1` and `x2`.

$$
c1 = \texttt{conv2d}\Bigg(
\begin{bmatrix}
    4 & 9 & 3 & 0 & 3 \\
    9 & 7 & 3 & 7 & 3 \\
    1 & 6 & 6 & 9 & 8 \\
    6 & 6 & 8 & 4 & 3 \\
    6 & 9 & 1 & 4 & 4 \\
\end{bmatrix},~
\begin{bmatrix}
    1 & 1 \\
    1 & 1
\end{bmatrix}\Bigg) = 
    \begin{bmatrix} 
        4+9+9+7 & 9+3+7+3 & 3+0+3+7 & 0+3+7+3 \\
        9+7+1+6 & 7+3+6+6 & 3+7+6+9 & 7+3+9+8  \\
        1+6+6+6 & 6+6+6+8 & 6+9+8+4 & 9+8+4+3 \\
        6+6+6+9 & 6+8+9+1 & 8+4+1+4 & 4+3+4+4 \\
    \end{bmatrix} =
\begin{bmatrix} 
        29 & 22 & 13 & 13 \\
        23 & 22 & 25 & 27  \\
        19 & 26 & 27 & 24 \\
        27 & 24 & 17 & 15 \\
\end{bmatrix}
$$

$$
c2 = \texttt{conv2d}\Bigg(
\begin{bmatrix}
    1 & 9 & 9 & 9 & 0 & 1 \\
    2 & 3 & 0 & 5 & 5 & 2 \\
    9 & 1 & 8 & 8 & 3 & 6 \\
    9 & 1 & 7 & 3 & 5 & 2 \\
    1 & 0 & 9 & 3 & 1 & 1 \\
    0 & 3 & 6 & 6 & 7 & 9 \\
\end{bmatrix},~
\begin{bmatrix}
    6 & 3 & 4 & 5 \\
    0 & 8 & 2 & 8 \\
    2 & 7 & 5 & 0 \\
    0 & 8 & 1 & 9 \\
\end{bmatrix}\Bigg) = \begin{bmatrix} 
    285 & 369 & 286 \\
    230 & 317 & 257 \\ 
    306 & 374 & 344 \\
\end{bmatrix}
$$


In [ ]:

torch.manual_seed(0)

def conv2d(img: torch.Tensor, kernel: torch.Tensor):
    """
    PARAMS
        img: the 2-dim image with a specific height and width
        kernel: a 2-dim kernel (smaller than image dimensions) that convolves the given image
    
    RETURNS
        the convolved 2-dim image
    """

    h, w = img.shape
    size = kernel.shape[0]
    stride = 1
    
    hp = int(math.floor((h - size)) + 1)
    wp = int(math.floor((w - size)) + 1)
    new_img = torch.zeros(size=(hp, wp))
    
    for i in range(0, h, stride):
        if i > h - size:
            break
        if i % stride == 0:
            for j in range(0, w, stride):
                if j > w - size:
                    break
                try:
                    if j % stride == 0:
                        out = (kernel * img[i:i+size, j:j+size]).sum()
                        new_img[i][j] = out
                except:
                    break
            
    return new_img


In [ ]:
# Try running these to see the convolution in action!
x1 = torch.tensor([
    [4, 9, 3, 0, 3],
    [9, 7, 3, 7, 3],
    [1, 6, 6, 9, 8],
    [6, 6, 8, 4, 3],
    [6, 9, 1, 4, 4]
])
k1 = torch.ones((2, 2))
x2 = torch.tensor([
    [1, 9, 9, 9, 0, 1],
    [2, 3, 0, 5, 5, 2],
    [9, 1, 8, 8, 3, 6],
    [9, 1, 7, 3, 5, 2],
    [1, 0, 9, 3, 1, 1],
    [0, 3, 6, 6, 7, 9]
])
k2 = torch.tensor([
    [6, 3, 4, 5],
    [0, 8, 2, 8],
    [2, 7, 5, 0],
    [0, 8, 1, 9]
])
c1 = conv2d(x1, k1)
c2 = conv2d(x2, k2)
print(c1, c2)

### What Pooling Does (Specifically Max Pooling)

**Pooling** is a downsampling operation typically applied after a convolutional layer. It takes small rectangular regions (e.g., 2x2) within the feature map and reduces each region down to a single output value.

* **Max Pooling:** The most common form, **Max Pooling**, selects only the **largest value** within the region to be the output. 

## Why We Do It

We use pooling in CNNs primarily to improve **efficiency** and **robustness**:

1.  **Dimensionality Reduction (Efficiency):** Pooling significantly **reduces the spatial size** (width and height) of the feature maps. This drastically cuts down the number of parameters and the computational load in subsequent layers.

2.  **Increased Robustness to Translation and Noise:** By only recording the presence of the most active feature (the max value), pooling makes the model slightly **invariant to small shifts or distortions** of the feature's location. If the feature moves a little bit within the pooling window, the output maximum value usually remains the same, ensuring the model still recognizes the feature.

***

Here we create a `maxpool2d` function that takes in an image, `img : torch.Tensor`, and a square kernel size `size : int`. We assume stride is 1 and there's no padding.

We can try it on two images `x1` and `x2` of different sizes.
$$
m1 = \texttt{maxpool2d}\Bigg(
\begin{bmatrix}
    4 & 9 & 3 & 0 & 3 \\
    9 & 7 & 3 & 7 & 3 \\
    1 & 6 & 6 & 9 & 8 \\
    6 & 6 & 8 & 4 & 3 \\
    6 & 9 & 1 & 4 & 4 \\
\end{bmatrix},~2\Bigg) =
\begin{bmatrix} 
        max(4,9,9,7) & max(9,3,7,3) & max(3,0,3,7) & max(0,3,7,3) \\
        max(9,7,1,6) & max(7,3,6,6) & max(3,7,6,9) & max(7,3,9,8)  \\
        max(1,6,6,6) & max(6,6,6,8) & max(6,9,8,4) & max(9,8,4,3) \\
        max(6,6,6,9) & max(6,8,9,1) & max(8,4,1,4) & max(4,3,4,4) \\
\end{bmatrix} =
\begin{bmatrix} 
        9 & 9 & 7 & 7 \\
        9 & 7 & 9 & 9  \\
        6 & 8 & 9 & 9 \\
        9 & 9 & 8 & 4 \\
\end{bmatrix}
$$

$$
m2 = \texttt{maxpool2d}\Bigg(
\begin{bmatrix}
    1 & 9 & 9 & 9 & 0 & 1 \\
    2 & 3 & 0 & 5 & 5 & 2 \\
    9 & 1 & 8 & 8 & 3 & 6 \\
    9 & 1 & 7 & 3 & 5 & 2 \\
    1 & 0 & 9 & 3 & 1 & 1 \\
    0 & 3 & 6 & 6 & 7 & 9 \\
\end{bmatrix},~3\Bigg) = \begin{bmatrix} 
    9 & 9 & 9 & 9 \\
    9 & 8 & 8 & 8 \\ 
    9 & 9 & 9 & 8 \\
    9 & 9 & 9 & 9 \\
\end{bmatrix}
$$

In [ ]:

torch.manual_seed(0)

def maxpool2d(img: torch.Tensor, size: int):
    """
    PARAMS
        img: the 2-dim image with a specific height and width
        size: an integer corresponding to the window size for Max Pooling
    
    RETURNS
        the 2-dim output after Max Pooling
    """

    h, w = img.shape
    stride = 1

    hp = int(math.floor((h - size)) + 1)
    wp = int(math.floor((w - size)) + 1)
    new_img = torch.zeros(size=(hp, wp))
    
    for i in range(0, h, stride):
        if i > h - size:
            break
        if i % stride == 0:
            for j in range(0, w, stride):
                if j > w - size:
                    break
                try:
                    out = torch.max(img[i:i+size, j:j+size])
                    new_img[i][j] = out
                except:
                    break
            
    return new_img


In [ ]:

x1 = torch.tensor([
    [4, 9, 3, 0, 3],
    [9, 7, 3, 7, 3],
    [1, 6, 6, 9, 8],
    [6, 6, 8, 4, 3],
    [6, 9, 1, 4, 4]
])
k1 = 2

x2 = torch.tensor([
    [1, 9, 9, 9, 0, 1],
    [2, 3, 0, 5, 5, 2],
    [9, 1, 8, 8, 3, 6],
    [9, 1, 7, 3, 5, 2],
    [1, 0, 9, 3, 1, 1],
    [0, 3, 6, 6, 7, 9]
])
k2 = 3


m1 = maxpool2d(x1, k1)
print(m1)
m2 = maxpool2d(x2, k2)
print(m2)